# LeetCode 1785 — Minimum Elements to Add to Form a Given Sum bằng Python

Link đề: https://leetcode.com/problems/minimum-elements-to-add-to-form-a-given-sum/ · Độ khó: **Medium** · Chủ đề: Array, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minElements(self, nums: List[int], limit: int, goal: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Rút gọn bài toán về **một con số**: khoảng cách `d = |goal − sum(nums)|` cần bù.
2. Hiểu vì sao mỗi phần tử thêm vào bù được **tối đa `limit`** và vì sao chọn "lớn nhất có thể" là tối ưu.
3. Viết vét cạn bằng **BFS trên giá trị tổng** để kiểm chứng công thức.
4. Tính **chia lấy trần** `⌈d / limit⌉` bằng số nguyên, tránh số thực và tràn số.
5. Nhận ra các bẫy: dấu của hiệu, `d = 0`, và số lớn (`|goal| ≤ 10⁹`, tổng tới `10¹¹`).

## 2. Đề bài & ví dụ

Cho mảng `nums` với tính chất `|nums[i]| ≤ limit`, và hai số nguyên `limit`, `goal`. Trả về số **phần tử ít nhất** cần **thêm** vào mảng để tổng bằng `goal`, sao cho mảng vẫn giữ tính chất `|x| ≤ limit` với mọi phần tử (kể cả phần tử thêm).

**Ví dụ 1:** `nums = [1,-1,1], limit = 3, goal = -4` → `2`. Tổng hiện tại 1; thêm `−2` và `−3` ⇒ `1 − 2 − 3 = −4`.

**Ví dụ 2:** `nums = [1,-10,9,1], limit = 100, goal = 0` → `1`. Tổng hiện tại 1; thêm `−1`.

**Ràng buộc:** `1 ≤ nums.length ≤ 10⁵`; `1 ≤ limit ≤ 10⁶`; `−limit ≤ nums[i] ≤ limit`; `−10⁹ ≤ goal ≤ 10⁹`.

In [1]:
import random
import time
import math
from collections import deque
from typing import List

examples = [([1, -1, 1], 3, -4, 2), ([1, -10, 9, 1], 100, 0, 1)]
for nums, limit, goal, exp in examples:
    print(f"nums={nums}, limit={limit}, goal={goal} | tổng={sum(nums)} -> kỳ vọng {exp}")

nums=[1, -1, 1], limit=3, goal=-4 | tổng=1 -> kỳ vọng 2
nums=[1, -10, 9, 1], limit=100, goal=0 | tổng=1 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 `abs` và hiệu cần bù

Cần thêm các phần tử có **tổng** `goal − sum(nums)`. Dấu chỉ cho biết thêm số dương hay âm; **độ lớn** `d = |goal − sum(nums)|` mới quyết định số lượng.

### 3.2 Chia lấy trần bằng số nguyên

`⌈a / b⌉ = (a + b − 1) // b` với `a ≥ 0`, `b > 0`; hoặc `−(−a // b)`.

In [2]:
for a, b in [(0, 3), (5, 3), (6, 3), (7, 3), (10 ** 11 + 1, 10 ** 6)]:
    print(a, b, "->", (a + b - 1) // b, -(-a // b))

0 3 -> 0 0
5 3 -> 2 2
6 3 -> 2 2
7 3 -> 3 3
100000000001 1000000 -> 100001 100001


### 3.3 Vì sao tránh `math.ceil(a / b)`

`a / b` là `float` (53 bit định trị). Với `a` lớn, phép chia có thể làm tròn sai, khiến `ceil` lệch 1.

In [3]:
a, b = 10 ** 17 + 1, 1
print("ceil số thực:", math.ceil(a / b), "| số nguyên:", (a + b - 1) // b)

ceil số thực: 100000000000000000 | số nguyên: 100000000000000001


### 3.4 BFS trên đồ thị "giá trị"

Đỉnh = giá trị tổng hiện tại; cạnh = thêm một số `x ∈ [−limit, limit]`. BFS cho số bước ít nhất từ `sum(nums)` tới `goal` — đúng định nghĩa đề (dùng để kiểm chứng với số nhỏ).

### 3.5 Bất đẳng thức tam giác cho tổng

`|x_1 + x_2 + … + x_k| ≤ |x_1| + |x_2| + … + |x_k|`. Nếu mỗi `|x_i| ≤ L` thì tổng của `k` số có độ lớn tối đa `k·L` — đây là "cận dưới" then chốt của bài: muốn bù được `d` thì cần `k·L ≥ d`. Dấu bằng xảy ra khi mọi `x_i` **cùng dấu** và bằng `±L`.

In [4]:
rng = random.Random(0)
for _ in range(5):
    xs = [rng.randint(-3, 3) for _ in range(4)]
    print(xs, "| |tổng| =", abs(sum(xs)), "<= tổng |x| =", sum(map(abs, xs)), "<= k*L =", 4 * 3)

[3, 0, 3, 0] | |tổng| = 6 <= tổng |x| = 6 <= k*L = 12
[-3, -1, 1, 0] | |tổng| = 3 <= tổng |x| = 5 <= k*L = 12
[0, 3, 3, -1] | |tổng| = 5 <= tổng |x| = 7 <= k*L = 12
[0, -1, 1, -2] | |tổng| = 2 <= tổng |x| = 4 <= k*L = 12
[1, -2, -1, -2] | |tổng| = 4 <= tổng |x| = 6 <= k*L = 12


## 4. Ý tưởng nền tảng / trực giác

**Ví dụ 1:** tổng 1, cần tới −4 ⇒ phải bù `−5`. Mỗi phần tử thêm vào có giá trị trong `[−3, 3]` ⇒ bù tối đa `3` (về độ lớn) mỗi lần.
- 1 phần tử: tối đa 3 < 5 ⇒ không đủ.
- 2 phần tử: tối đa 6 ≥ 5 ⇒ đủ (ví dụ `−3` và `−2`).

⇒ đáp án `⌈5 / 3⌉ = 2`.

**Ví dụ 2:** tổng 1, goal 0 ⇒ bù `−1` ⇒ `⌈1/100⌉ = 1`.

Trực giác "đổ nước bằng gáo dung tích `limit`": cần bao nhiêu gáo để đổ `d` lít? `⌈d / limit⌉` gáo (gáo cuối có thể không đầy).

In [5]:
for nums, limit, goal, exp in examples:
    d = abs(goal - sum(nums))
    print(f"d = |{goal} - {sum(nums)}| = {d}, limit = {limit} -> ceil = {(d + limit - 1) // limit} (kỳ vọng {exp})")

d = |-4 - 1| = 5, limit = 3 -> ceil = 2 (kỳ vọng 2)
d = |0 - 1| = 1, limit = 100 -> ceil = 1 (kỳ vọng 1)


## 5. Cách 1 — Brute Force / baseline: BFS trên giá trị tổng (chỉ cho số nhỏ)

In [6]:
def min_elements_bfs(nums, limit, goal):
    start = sum(nums)
    lo, hi = min(start, goal) - limit, max(start, goal) + limit        # không cần đi quá xa hai đầu
    dist = {start: 0}; q = deque([start])
    while q:
        v = q.popleft()
        if v == goal:
            return dist[v]
        for x in range(-limit, limit + 1):
            w = v + x
            if lo <= w <= hi and w not in dist:
                dist[w] = dist[v] + 1; q.append(w)
    return -1

for nums, limit, goal, exp in examples:
    got = min_elements_bfs(nums, limit, goal); print(got, "kỳ vọng", exp); assert got == exp

2 kỳ vọng 2
1 kỳ vọng 1


**Độ phức tạp:** `O(D · limit)` với `D = |goal − sum|` — với `D` tới `2·10⁹` là **không thể**. Nhưng nó bám sát định nghĩa nên là đáp án chuẩn cho test nhỏ.

**Nút thắt:** BFS thử mọi giá trị phần tử, trong khi rõ ràng mỗi bước nên "đi xa nhất" về phía `goal`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** sau `k` phần tử thêm vào, tổng thay đổi tối đa `k · limit` (về độ lớn) ⇒ cần `k · limit ≥ d` ⇒ `k ≥ ⌈d / limit⌉` (**cận dưới**). Và `k = ⌈d / limit⌉` **đạt được**: thêm `k − 1` phần tử bằng `±limit` (cùng dấu với `goal − sum`) và một phần tử cuối bằng phần dư (`|phần dư| ≤ limit`).

**Quy tắc:**
- **Nếu** `d = 0` **thì** không cần thêm gì (`⌈0/limit⌉ = 0`).
- **Ngược lại** đáp án là `⌈d / limit⌉`.

Tham lam "mỗi bước bù `min(limit, phần còn lại)`" cho cùng kết quả — là cách mô phỏng công thức.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng tham lam từng phần tử

In [7]:
def min_elements_greedy(nums, limit, goal):
    rem = goal - sum(nums); cnt = 0
    while rem != 0:
        step = max(-limit, min(limit, rem))          # bù nhiều nhất có thể về phía goal
        rem -= step; cnt += 1
    return cnt

print([min_elements_greedy(n, l, g) for n, l, g, _ in examples])

[2, 1]


**Độ phức tạp Cách 2:** `O(n + d / limit)` — tệ nhất `d / limit ≈ 2·10¹¹ / 1`? (khi `limit = 1`, `|goal − sum| ≤ 10⁹ + 10¹¹`) ⇒ **quá chậm**. Chỉ dùng để minh họa.

### 7.2 Cách 3 — Mô phỏng theo khối (gom nhiều bước `limit` một lúc)

Thay vì trừ từng `limit`, trừ `q = d // limit` khối một lần, rồi xử lý phần dư — chính là `divmod`.

In [8]:
def min_elements_divmod(nums, limit, goal):
    d = abs(goal - sum(nums))
    q, r = divmod(d, limit)
    return q + (r > 0)

print([min_elements_divmod(n, l, g) for n, l, g, _ in examples])

[2, 1]


**Độ phức tạp Cách 3:** `O(n)` (tính tổng). Biểu thức `q + (r > 0)` dùng `True == 1`.

### 7.3 Cách 4 — Công thức chia lấy trần

In [9]:
def min_elements_formula(nums, limit, goal):
    d = abs(goal - sum(nums))
    return (d + limit - 1) // limit

print([min_elements_formula(n, l, g) for n, l, g, _ in examples])

[2, 1]


**Code nộp LeetCode (Cách 4):** xem mục 9. **Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### 7.4 Cách 5 — Xây dựng **chính các phần tử** cần thêm (để kiểm chứng tính đạt được)

In [10]:
def elements_to_add(nums, limit, goal):
    rem = goal - sum(nums); sign = 1 if rem > 0 else -1
    d = abs(rem); q, r = divmod(d, limit)
    return [sign * limit] * q + ([sign * r] if r else [])

for nums, limit, goal, _ in examples:
    add = elements_to_add(nums, limit, goal)
    print(add, "| tổng mới:", sum(nums) + sum(add), "| mọi |x| <= limit:", all(abs(x) <= limit for x in add))

[-3, -2] | tổng mới: -4 | mọi |x| <= limit: True
[-1] | tổng mới: 0 | mọi |x| <= limit: True


## 8. TRACE từng bước

Trace Cách 2 (tham lam) và Cách 4 trên **Ví dụ 1**:

| bước | phần còn lại `rem` | thêm | `rem` mới |
|---|---|---|---|
| 1 | −5 | −3 | −2 |
| 2 | −2 | −2 | 0 |

Công thức: `d = 5`, `(5 + 3 − 1) // 3 = 7 // 3 = 2`.

In [11]:
def trace_greedy(nums, limit, goal):
    rem = goal - sum(nums); cnt = 0
    print(f"tổng={sum(nums)}, goal={goal} -> cần bù {rem}")
    while rem != 0:
        step = max(-limit, min(limit, rem)); rem -= step; cnt += 1
        print(f"  bước {cnt}: thêm {step:>4} -> còn {rem}")
    d = abs(goal - sum(nums))
    print(f"công thức: ({d} + {limit} - 1) // {limit} = {(d + limit - 1) // limit}")
    return cnt

assert trace_greedy(*examples[0][:3]) == 2
print()
assert trace_greedy(*examples[1][:3]) == 1

tổng=1, goal=-4 -> cần bù -5
  bước 1: thêm   -3 -> còn -2
  bước 2: thêm   -2 -> còn 0
công thức: (5 + 3 - 1) // 3 = 2

tổng=1, goal=0 -> cần bù -1
  bước 1: thêm   -1 -> còn 0
công thức: (1 + 100 - 1) // 100 = 1


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** — công thức chia lấy trần.

In [12]:
class Solution:
    def minElements(self, nums: List[int], limit: int, goal: int) -> int:
        d = abs(goal - sum(nums))
        return (d + limit - 1) // limit

for nums, limit, goal, exp in examples:
    got = Solution().minElements(nums, limit, goal); print(got, exp); assert got == exp

2 2
1 1


## 10. Vì sao thuật toán đúng

Đặt `D = goal − sum(nums)`, `d = |D|`.

**Cận dưới.** Nếu thêm `k` phần tử `x_1, …, x_k` với `|x_i| ≤ limit` và `Σ x_i = D`, thì `d = |Σ x_i| ≤ Σ |x_i| ≤ k · limit` (bất đẳng thức tam giác) ⇒ `k ≥ d / limit` ⇒ `k ≥ ⌈d / limit⌉` (vì `k` nguyên).

**Đạt được.** Với `k = ⌈d/limit⌉`, chọn `q = d // limit` phần tử `sign(D)·limit` và (nếu `r = d mod limit > 0`) một phần tử `sign(D)·r`; tổng đúng bằng `D`, mọi phần tử thỏa `|x| ≤ limit`, số phần tử `= q + [r > 0] = ⌈d/limit⌉`. ∎

Mảng ban đầu vẫn giữ tính chất vì ta không sửa phần tử cũ.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `d = |goal − sum|` (có thể tới `10⁹ + 10¹¹`).

| Cách | Time | Space |
|---|---|---|
| 1. BFS trên giá trị | `O(d · limit)` | `O(d)` |
| 2. Tham lam từng phần tử | `O(n + d/limit)` | `O(1)` |
| 3. `divmod` | `O(n)` | `O(1)` |
| 4. Công thức trần | `O(n)` | `O(1)` |
| 5. Dựng phần tử | `O(n + d/limit)` (kích thước output) | `O(d/limit)` |

In [13]:
rng = random.Random(1785)
nums = [rng.randint(-10 ** 6, 10 ** 6) for _ in range(10 ** 5)]
for name, f in [("divmod", min_elements_divmod), ("formula", Solution().minElements)]:
    t = time.perf_counter(); r = f(nums, 10 ** 6, -10 ** 9); print(f"{name:<8} -> {r} ({(time.perf_counter()-t)*1000:.1f} ms)")
t = time.perf_counter(); r = min_elements_greedy(nums, 10 ** 6, -10 ** 9); print(f"greedy   -> {r} ({(time.perf_counter()-t)*1000:.1f} ms, vì limit lớn nên số bước nhỏ)")
print("tham lam với limit = 1 sẽ cần ~", abs(-10 ** 9 - sum([1] * 10 ** 5)), "bước -> không khả thi")

divmod   -> 1115 (0.8 ms)
formula  -> 1115 (0.6 ms)
greedy   -> 1115 (0.6 ms, vì limit lớn nên số bước nhỏ)
tham lam với limit = 1 sẽ cần ~ 1000100000 bước -> không khả thi


## 12. Bộ test

In [14]:
ALL = {"greedy": min_elements_greedy, "divmod": min_elements_divmod, "formula": min_elements_formula,
       "submit": Solution().minElements}
def run_all(nums, limit, goal, with_bfs=False):
    res = {k: f(nums, limit, goal) for k, f in ALL.items()}
    if with_bfs:
        res["bfs"] = min_elements_bfs(nums, limit, goal)
    assert len(set(res.values())) == 1, (nums, limit, goal, res)
    add = elements_to_add(nums, limit, goal)
    assert len(add) == res["formula"] and sum(nums) + sum(add) == goal and all(abs(x) <= limit for x in add)
    return res["formula"]

for nums, limit, goal, exp in examples:
    assert run_all(nums, limit, goal, with_bfs=True) == exp
edge = [([0], 1, 0, 0), ([5], 5, 5, 0), ([1], 1, -1, 2), ([1], 1, 1000, 999), ([-3], 3, 3, 2), ([2, 2], 2, 0, 2)]
for nums, limit, goal, exp in edge:
    assert run_all(nums, limit, goal, with_bfs=True) == exp, (nums, limit, goal)
rng = random.Random(17850)
for _ in range(400):                                    # có BFS (số nhỏ)
    limit = rng.randint(1, 6); nums = [rng.randint(-limit, limit) for _ in range(rng.randint(1, 5))]
    run_all(nums, limit, rng.randint(-30, 30), with_bfs=True)
for _ in range(400):                                    # số lớn, bỏ tham lam từng bước khi quá chậm
    limit = rng.randint(1, 10 ** 6); nums = [rng.randint(-limit, limit) for _ in range(rng.randint(1, 50))]
    goal = rng.randint(-10 ** 9, 10 ** 9)
    assert min_elements_divmod(nums, limit, goal) == Solution().minElements(nums, limit, goal) == min_elements_formula(nums, limit, goal)
print("Ví dụ, edge case, 400 test có BFS, 400 test số lớn: OK")

Ví dụ, edge case, 400 test có BFS, 400 test số lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng chia nguyên thường (`//`) thay vì chia lấy trần

In [15]:
def wrong_floor(nums, limit, goal):
    return abs(goal - sum(nums)) // limit

for nums, limit, goal, exp in examples:
    print("sai :", wrong_floor(nums, limit, goal), "| đúng:", exp)

sai : 1 | đúng: 2
sai : 0 | đúng: 1


**Vì sao sai:** phần dư (nếu có) vẫn cần thêm **một** phần tử. **Sửa:** `(d + limit − 1) // limit`.

### Lỗi 2 — Quên `abs` (hiệu âm)

In [16]:
def wrong_no_abs(nums, limit, goal):
    d = goal - sum(nums)
    return (d + limit - 1) // limit

nums, limit, goal, exp = examples[0]
print("sai :", wrong_no_abs(nums, limit, goal), "| đúng:", exp)

sai : -1 | đúng: 2


**Vì sao sai:** `d = −5` ⇒ `(−5 + 2) // 3 = −1` — vô nghĩa. Ta có thể thêm số âm nên chỉ độ lớn quan trọng. **Sửa:** `abs`.

### Lỗi 3 — `math.ceil` với số thực ở giá trị lớn

In [17]:
def wrong_float(nums, limit, goal):
    return math.ceil(abs(goal - sum(nums)) / limit)

nums = [10 ** 6] * 10 ** 5; limit = 10 ** 6; goal = -10 ** 9      # tổng 10^11, d = 1.01e11
print("số thực:", wrong_float(nums, limit, goal), "| số nguyên:", Solution().minElements(nums, limit, goal))
big_d = 3 * 10 ** 16 + 1
print("d lớn hơn 2^53 (minh họa):", math.ceil(big_d / 3), "vs", (big_d + 2) // 3)

số thực: 101000 | số nguyên: 101000
d lớn hơn 2^53 (minh họa): 10000000000000000 vs 10000000000000001


**Vì sao sai (tiềm ẩn):** trong giới hạn đề, `d ≤ 1,01·10¹¹ < 2⁵³` nên `float` vẫn đúng — nhưng thói quen này hỏng ngay khi số vượt `2⁵³` (dòng thứ hai). Trong C++/Java còn phải lo **tràn `int`** khi cộng tổng (`10¹¹ > 2³¹`). **Sửa:** chia nguyên, dùng `long` ở ngôn ngữ khác.

### Lỗi 4 — Nghĩ rằng phải giữ phần tử thêm vào **khác nhau** / dương

In [18]:
def wrong_positive_only(nums, limit, goal):
    d = goal - sum(nums)
    if d < 0:
        return -1                                 # SAI: được phép thêm số âm
    return (d + limit - 1) // limit

nums, limit, goal, exp = examples[0]
print("sai :", wrong_positive_only(nums, limit, goal), "| đúng:", exp)

sai : -1 | đúng: 2


**Vì sao sai:** ràng buộc chỉ là `|x| ≤ limit`; số âm hoàn toàn hợp lệ (Ví dụ 1 thêm `−2`, `−3`). **Sửa:** dùng `abs`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Ít phần tử nhất, mỗi phần tử đóng góp tối đa `L`, cần tổng `d`" ⇒ `⌈d / L⌉`.
- Luôn chứng minh hai chiều: **cận dưới** (không thể ít hơn) + **xây dựng** (đạt được).
- Chia lấy trần bằng số nguyên; nhớ `abs` và trường hợp `d = 0`.
- Bài "Medium" nhưng ngắn: điểm khó nằm ở đọc đề và các bẫy số học.

## 15. Biến thể của bài

1. **Phần tử thêm vào phải nằm trong `[a, b]` với `a ≥ 0`** (chỉ dương): nếu `D < 0` thì `−1` (hoặc nếu `a > 0` có ràng buộc tổng tối thiểu) — Bài tập 5.
2. **Mỗi phần tử thêm vào có chi phí `|x|`**: tổng chi phí luôn `≥ d` và đạt `d` với bất kỳ cách chia hợp lệ.
3. **2335 Minimum Amount of Time to Fill Cups**: số bước khi mỗi bước lấy từ hai nhóm.
4. **1658 Minimum Operations to Reduce X to Zero**: cũng "đạt tổng", nhưng chọn từ hai đầu mảng (cửa sổ trượt).
5. **Liệt kê các phần tử cần thêm** (Cách 5 / Bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS giá trị | Định nghĩa thuần | `O(d·limit)` | `O(d)` | Kiểm chứng số nhỏ |
| 2. Tham lam từng bước | Bù tối đa mỗi lần | `O(n + d/limit)` | `O(1)` | Minh họa |
| 3. `divmod` | Thương + có dư | `O(n)` | `O(1)` | Rõ nghĩa |
| 4. Công thức trần | `(d+L−1)//L` | `O(n)` | `O(1)` | **Bản nộp** |
| 5. Dựng phần tử | Xây lời giải cụ thể | `O(d/limit)` | `O(d/limit)` | Khi cần output phần tử |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([0], 5, 0)`; (b) `([3], 3, -3)`; (c) `([1,2,3], 2, 0)`; (d) `([-1], 1, 10)`.

**Bài 2 (tính tay):** Với `nums = [5, -2, 7]`, `limit = 4`, `goal = -9`: tính `d`, đáp án, và một tập phần tử cần thêm.

**Bài 3 (điền TODO):** Hoàn thiện `ceil_div(a, b)` (a ≥ 0, b > 0) không dùng số thực.

**Bài 4 (tự viết lại):** Viết `min_elements_list(nums, limit, goal)` trả list các phần tử cần thêm (số lượng tối thiểu), không dùng `divmod`.

**Bài 5 (mini challenge):** Phần tử thêm vào phải thuộc đoạn `[lo, hi]` với `lo ≤ hi` (có thể cùng dấu). Viết `min_elements_range(nums, lo, hi, goal)` trả số phần tử ít nhất hoặc `−1`; đối chiếu BFS.

In [19]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def ceil_div_todo(a, b):
    # TODO: trả (a + b - 1) // b
    pass

In [20]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def min_elements_list(nums, limit, goal):
    # TODO: vòng while: lấy step = max(-limit, min(limit, rem)), thêm vào list
    pass

def min_elements_range(nums, lo, hi, goal):
    # TODO: với k phần tử, tổng đạt được là mọi số nguyên trong [k*lo, k*hi]; tìm k nhỏ nhất chứa D
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `d = |goal − sum|`. Mức 2: `⌈d/limit⌉`. Mức 3: (a) `0`; (b) `2`; (c) `3`; (d) `11`.

**Bài 2.** Mức 1: tổng `10`, `D = −19`, `d = 19`. Mức 2: `⌈19/4⌉ = 5`. Mức 3: thêm `−4, −4, −4, −4, −3`.

**Bài 3.** Mức 1: `a // b` là sàn. Mức 2: cộng `b − 1` trước khi chia. Mức 3: kiểm tra với `a = 0`, `a` chia hết `b`.

**Bài 4.** Mức 1: `rem = goal − sum`. Mức 2: mỗi bước `step = max(−limit, min(limit, rem))`. Mức 3: độ dài list phải bằng công thức.

**Bài 5.** Mức 1: với `k` phần tử, tổng thêm vào là mọi số trong `[k·lo, k·hi]`. Mức 2: `k = 0` chỉ cho `D = 0`. Mức 3: duyệt `k` từ 0 tới giới hạn hợp lý (BFS đối chiếu); nếu `lo > 0` và `D < 0` thì `−1`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
b1 = [Solution().minElements(*a) for a in [([0], 5, 0), ([3], 3, -3), ([1, 2, 3], 2, 0), ([-1], 1, 10)]]
print("Bài 1:", b1); assert b1 == [0, 2, 3, 11]
print("Bài 2:", Solution().minElements([5, -2, 7], 4, -9), elements_to_add([5, -2, 7], 4, -9))
assert Solution().minElements([5, -2, 7], 4, -9) == 5
def ceil_div_todo(a, b):
    return (a + b - 1) // b
assert all(ceil_div_todo(a, b) == -(-a // b) for a in range(200) for b in range(1, 20))
print("Bài 3: OK")

Bài 1: [0, 2, 3, 11]
Bài 2: 5 [-4, -4, -4, -4, -3]
Bài 3: OK


In [22]:
# Bài 4, 5
def min_elements_list(nums, limit, goal):
    rem = goal - sum(nums); out = []
    while rem:
        step = max(-limit, min(limit, rem)); out.append(step); rem -= step
    return out

def min_elements_range(nums, lo, hi, goal):
    D = goal - sum(nums)
    if D == 0:
        return 0
    # k*lo <= D <= k*hi với k >= 1 nhỏ nhất
    best = None
    for k in range(1, abs(D) + 2 + abs(lo) + abs(hi)):
        if k * lo <= D <= k * hi:
            best = k; break
    return -1 if best is None else best

def range_bfs(nums, lo, hi, goal, cap=60):
    start = sum(nums); dist = {start: 0}; q = deque([start])
    while q:
        v = q.popleft()
        if v == goal: return dist[v]
        if dist[v] >= cap: continue
        for x in range(lo, hi + 1):
            w = v + x
            if abs(w - goal) <= 200 and w not in dist:
                dist[w] = dist[v] + 1; q.append(w)
    return -1

rng = random.Random(4)
for _ in range(300):
    limit = rng.randint(1, 5); nums = [rng.randint(-limit, limit) for _ in range(3)]; goal = rng.randint(-25, 25)
    lst = min_elements_list(nums, limit, goal)
    assert len(lst) == Solution().minElements(nums, limit, goal) and sum(nums) + sum(lst) == goal
    lo = rng.randint(-4, 4); hi = rng.randint(lo, 5)
    assert min_elements_range(nums, lo, hi, goal) == range_bfs(nums, lo, hi, goal), (nums, lo, hi, goal)
print("Bài 4, 5: OK |", min_elements_list([1, -1, 1], 3, -4), min_elements_range([0], 2, 3, 7), min_elements_range([0], 2, 3, -1))

Bài 4, 5: OK | [-3, -2] 3 -1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
d = abs(goal - sum(nums))
return (d + limit - 1) // limit
```

**Checklist:**
- [ ] Dùng `abs` (có thể thêm số âm)?
- [ ] Chia **lấy trần** bằng số nguyên?
- [ ] `d = 0` trả `0`?
- [ ] Ở C++/Java dùng `long` cho tổng?
- [ ] Đã test `limit = 1`, `goal` xa (`±10⁹`), tổng đã bằng `goal`?